# Skripsi: Integrasi Confident Learning dan CORN pada IndoBERT
## Pilot Study — Deteksi Label Noise pada Prediksi Rating Ulasan Google Play Store

**Ryan Besto Saragih — NIM 23051204205**
Pembimbing: Anita Qoiriah, S.Kom., M.Kom.

Notebook ini menjalankan pipeline lengkap pilot study sesuai Bab 1 (Subbab 1.2) dan
Bab 3 proposal: scraping → praproses → ablasi proxy classifier (P1-P4) →
deteksi noise → validasi manusia → training 6 skenario model → uji signifikansi
statistik (4 hipotesis pre-registered).

**Struktur data**: satu skema tunggal (tidak ada versioning) — `all_reviews_master.csv`
dikunci setelah scraping selesai dan dipakai sebagai sumber kebenaran tunggal untuk
seluruh eksperimen berikutnya, sesuai Batasan Masalah Bab 1.6.

**Cara pakai**: jalankan cell berurutan dari atas ke bawah. Cell scraping (Cell 3)
otomatis dilewati kalau data sudah ada — jangan hapus `all_reviews_master.csv`
kecuali memang berniat scraping ulang dari nol.

In [ ]:
# ==========================================================
# CELL 2: MOUNT DRIVE + CLONE/PULL REPO
# ==========================================================
from google.colab import drive
import os, sys

drive.mount('/content/drive')

REPO_URL = "https://github.com/bestoism/skripsi-corn-label-noise"
REPO_DIR = "/content/skripsi-corn-label-noise"

if os.path.exists(REPO_DIR):
    print("🔄 Repo sudah ada, menarik update terbaru...")
    !cd {REPO_DIR} && git pull
else:
    print("⬇️  Clone repo baru...")
    !git clone {REPO_URL} {REPO_DIR}

sys.path.append(REPO_DIR)
os.chdir(REPO_DIR)
print(f"\n✅ Setup selesai. Working dir: {os.getcwd()}")

Mounted at /content/drive
⬇️  Clone repo baru...
Cloning into '/content/skripsi-corn-label-noise'...
remote: Enumerating objects: 109, done.
remote: Counting objects: 100% (109/109), done.
remote: Compressing objects: 100% (67/67), done.
remote: Total 109 (delta 63), reused 80 (delta 37), pack-reused 0 (from 0)
Receiving objects: 100% (109/109), 348.82 KiB | 1.39 MiB/s, done.
Resolving deltas: 100% (63/63), done.

✅ Setup selesai. Working dir: /content/skripsi-corn-label-noise


## Instalasi Dependencies

Termasuk `py3langid` untuk filter bahasa saat scraping. Pastikan `requirements.txt`
di repo sudah memuat: `pandas`, `numpy`, `scikit-learn`, `torch`, `transformers`,
`cleanlab`, `coral-pytorch`, `statsmodels`, `scipy`, `google-play-scraper`,
`py3langid`.

In [ ]:
# ==========================================================
# CELL 4: INSTALL REQUIREMENTS
# ==========================================================
!pip install -q -r requirements.txt
print("✅ Dependencies terpasang.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 306.1/306.1 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 18.5 MB/s eta 0:00:00
✅ Dependencies terpasang.


## Import Umum
Dipakai berulang di banyak cell berikutnya — diletakkan sekali di sini supaya
tidak diulang-ulang di tiap cell.

In [ ]:
# ==========================================================
# CELL 6: IMPORT UMUM
# ==========================================================
import os
import pandas as pd
import numpy as np

from src import config
print(f"📌 Proxy default aktif: [{config.PROXY_ID}] {config.PROXY_NAME}")
print(f"📁 Drive root: {config.DRIVE_ROOT}")

📌 Proxy aktif: [3] finetuned_corn — IndoBERT fine-tuned K-Fold, CORN loss (P4) -- DEFAULT/FINAL
📌 Proxy default aktif: [3] finetuned_corn
📁 Drive root: /content/drive/MyDrive/SKRIPSI_CORN


## Scraping Google Play Store

Scraping dengan filter bahasa Indonesia inline (`py3langid`), target per rating
sudah dinaikkan untuk mengantisipasi susut data sepanjang pipeline (praproses →
split → K-Fold → pruning), dan dilengkapi laporan kualitas data (distribusi
tanggal per app/rating untuk cek confound temporal, distribusi panjang teks,
deteksi teks duplikat) yang otomatis tersimpan di `results/` untuk dilampirkan
di Bab III dan Bab IV.

⚠️ **JALANKAN SEKALI SAJA SEUMUR PROYEK.** Setelah selesai, `all_reviews_master.csv`
dikunci dan dipakai sebagai sumber kebenaran tunggal. Cell ini **otomatis dilewati**
kalau file tersebut sudah ada — hapus manual hanya kalau memang sengaja ingin
scraping ulang dari nol (dan sadar semua eksperimen berikutnya perlu diulang).

Estimasi waktu: bisa 1-3 jam tergantung koneksi dan seberapa jauh Google Play
Store perlu ditelusuri untuk rating 2/3 yang jarang muncul.

In [ ]:
# ==========================================================
# CELL 8: SCRAPING GOOGLE PLAY STORE (AUTO-SKIP KALAU SUDAH ADA)
# ==========================================================
from src import config

if os.path.exists(config.RAW_DATA_FILE):
    print(f"✅ {config.RAW_DATA_FILE} sudah ada -- scraping dilewati.")
    print("   Hapus file ini manual kalau memang mau scraping ulang dari nol.")
    df_check = pd.read_csv(config.RAW_DATA_FILE)
    print(f"   Baris tersedia: {len(df_check)}")
else:
    from scripts.scrape_google_play import main as run_scraping
    run_scraping()

## Praproses Teks

Case folding, penghapusan URL/username, translasi emoji ke sentimen (kamus
diperluas), normalisasi elongasi huruf, normalisasi slang (Kamus Alay + kamus
domain, dengan perbaikan pencocokan token bertanda-baca), resolusi konflik
teks-identik-rating-berbeda (voting mayoritas), dan penghapusan duplikat murni.

Preprocessing dijaga minimal (tanpa stemming/stopword removal) sesuai rekomendasi
untuk model berbasis BERT (Wilie dkk. 2020; Koto dkk. 2020).

⚠️ **JALANKAN SEKALI SAJA.** Split train/test yang dihasilkan (`split_train_raw.csv`
/ `split_test.csv`) dikunci dan dipakai di seluruh eksperimen berikutnya, termasuk
pilot study proxy P1-P4. Cell ini otomatis dilewati kalau split sudah ada.

In [ ]:
# ==========================================================
# CELL 10: PREPROCESSING + TRAIN-TEST SPLIT (SUMBER KEBENARAN TUNGGAL)
# ==========================================================
from sklearn.model_selection import train_test_split
from src.preprocess import run_preprocessing
from src import config

if os.path.exists(config.TRAIN_RAW_FILE) and os.path.exists(config.TEST_FILE):
    print("✅ Split train/test sudah ada -- preprocessing dilewati.")
    df_train = pd.read_csv(config.TRAIN_RAW_FILE)
    df_test = pd.read_csv(config.TEST_FILE)
    print(f"   Train: {len(df_train)} baris | Test: {len(df_test)} baris")
else:
    print("=" * 60)
    print(" PREPROCESSING ")
    print("=" * 60)
    df_clean = run_preprocessing(config.RAW_DATA_FILE, config.CLEAN_TEXT_FILE)
    df_clean = df_clean.dropna(subset=["cleaned_text", "rating"])

    print("\n" + "=" * 60)
    print(" SPLIT DATA (80% train, 20% test, stratified by rating, seed=42) ")
    print("=" * 60)
    df_train, df_test = train_test_split(
        df_clean, test_size=0.2, random_state=42, stratify=df_clean["rating"]
    )
    df_train.to_csv(config.TRAIN_RAW_FILE, index=False)
    df_test.to_csv(config.TEST_FILE, index=False)

    print(f"✅ Train: {len(df_train)} baris -> {config.TRAIN_RAW_FILE}")
    print(f"✅ Test : {len(df_test)} baris -> {config.TEST_FILE}")

📖 Kamus slang dasar: 15006 entri (Salsabila dkk., 2018)
 PREPROCESSING 
📥 Membaca data mentah dari: /content/drive/MyDrive/SKRIPSI_CORN/data/raw/all_reviews_master.csv
📏 Jumlah data awal: 16418 baris
🧹 Cleaning teks (lowercase, URL/tag, emoji, elongasi, slang)...
📊 Cakupan kamus slang (estimasi dari raw text): 29374/261962 kata (11.21%)
📏 Distribusi panjang teks: median=10 kata, ulasan sangat pendek (<=3 kata): 20.35%
🔍 Teks identik, rating berbeda: 144 kasus
🗑️  Baris dibuang (rating minoritas dalam grup konflik): 841
🗑️  Baris dibuang (tie, tidak ada mayoritas jelas): 181
   Total dibuang: 1022
✅ Setelah dibersihkan: 14219 baris (terbuang: 2199)
💾 Disimpan di: /content/drive/MyDrive/SKRIPSI_CORN/data/processed/reviews_clean.csv

 SPLIT DATA (80% train, 20% test, stratified by rating, seed=42) 
✅ Train: 11375 baris -> /content/drive/MyDrive/SKRIPSI_CORN/data/processed/split_train_raw.csv
✅ Test : 2844 baris -> /content/drive/MyDrive/SKRIPSI_CORN/data/processed/split_test.csv


In [ ]:
# ==========================================================
# CELL 10B: SAMPLING SUBSET UJI EMAS (GOLD TEST SUBSET) -- Subbab 3.9.4
# ==========================================================
from src import config, gold_test

if os.path.exists(config.GOLD_TEST_SAMPLE_FILE):
    print(f"✅ {config.GOLD_TEST_SAMPLE_FILE} sudah ada -- sampling dilewati.")
else:
    gold_test.sample_gold_test_subset(seed=42)
    gold_test.export_second_annotator_gold_subset()

print("\n⚠️  Anotasi gold test bisa dikerjakan PARALEL sambil menunggu training")
print("   6 skenario (Cell 20) berjalan -- evaluasinya baru dijalankan di Cell 23")
print("   setelah training tuntas. Isi 'human_gold_rating' (1-5) langsung di file di atas.")

📐 Populasi data uji: 2844 baris
   Target Cochran (e=10%): 93 baris
   Ukuran sampel dipakai: 93 baris

📝 Sample uji emas (BUTA): 93 baris -> /content/drive/MyDrive/SKRIPSI_CORN/gold_test/gold_test_sample.csv
   -> Isi kolom 'human_gold_rating' manual (1-5) langsung di file yang sama,
      lalu jalankan gold_test.evaluate_on_gold_subset() setelah training
      6 skenario (Cell 20) dan uji signifikansi (Cell 22) selesai.
📝 Sample anotator kedua (uji emas): 28 baris -> /content/drive/MyDrive/SKRIPSI_CORN/gold_test/gold_test_sample_annotator2.csv

⚠️  Anotasi gold test bisa dikerjakan PARALEL sambil menunggu training
   6 skenario (Cell 20) berjalan -- evaluasinya baru dijalankan di Cell 23
   setelah training tuntas. Isi 'human_gold_rating' (1-5) langsung di file di atas.


Ablasi Proxy Classifier (P1-P4) -- Subbab 3.6.1
Menjalankan Confident Learning untuk 4 dari 5 tahap ablasi proxy classifier (Tabel 3.1, Bab III): P1 (CLS embedding beku + LR), P2 (mean-pooling beku + LR), P3 (IndoBERT fine-tuned, Cross-Entropy), P4 (IndoBERT fine-tuned, CORN -- proxy final). P5 (fusi sentimen eksternal) dijalankan terpisah di Cell berikutnya karena butuh model sentimen tambahan.

Ini BUKAN bagian dari pilot study terpisah -- ini adalah pelaksanaan langsung studi ablasi lima tahap yang direncanakan di Bab III Subbab 3.6.1, dilaporkan lengkap di Bab IV.

Setiap proxy sekarang: (a) memakai skema K-Fold yang identik (shuffle=True, seed=42) untuk P1/P2 maupun P3/P4, (b) dikalibrasi lewat temperature scaling untuk P3/P4, (c) metode filter cleanlab dipilih otomatis lewat kriteria eksplisit (paling dekat dengan estimasi teoretis num_label_issues() versi off_diagonal_calibrated -- BUKAN off_diagonal default, karena off_diagonal identik secara matematis dengan hitungan confident_learning sehingga tidak pernah diskriminatif).

Hasilnya terkumpul di results/proxy_ablation_table.csv -- inilah Tabel 3.1 di Bab III/IV proposal.

⚠️ PENTING: kalau ini adalah run ULANG setelah perbaikan _select_filter_method, PERHATIKAN apakah metode terpilih untuk P4 (proxy final) berubah dari run sebelumnya. Kalau berubah, WAJIB hapus experiment_progress.json sebelum Cell 20, karena auto-resume tidak tahu data cleaned di baliknya sudah berubah.

⏱️ P3 dan P4 butuh fine-tuning K-Fold (5 fold × 3 epoch masing-masing) -- cell ini bisa memakan waktu cukup lama.

In [ ]:
# ==========================================================
# CELL 12: ABLASI PROXY CLASSIFIER P1-P4 (Subbab 3.6.1, Tabel 3.1)
# ==========================================================
import pandas as pd
from src import config
from src.clean import run_confident_learning

PROXY_IDS_TO_RUN = [0, 1, 2, 3]  # P5 (fusion, id 4) dijalankan terpisah di Cell 14

for pid in PROXY_IDS_TO_RUN:
    print(f"\n{'='*70}\n ABLASI PROXY CLASSIFIER -- PROXY_ID = {pid} \n{'='*70}")
    config.set_proxy(pid)
    try:
        run_confident_learning()
    except Exception as e:
        print(f"⚠️ Proxy {pid} gagal: {e}")
        continue

print("\n✅ Ablasi proxy P1-P4 selesai.")
proxy_table = pd.read_csv(config.PROXY_QUALITY_LOG_FILE)
display(proxy_table)

p4_row = proxy_table[proxy_table["proxy_id"] == 3]
if not p4_row.empty:
    print(f"\n🔍 Metode terpilih untuk proxy final (P4): {p4_row['selected_filter_method'].values[0]}")
    print("   Kalau ini beda dari run sebelumnya, HAPUS experiment_progress.json")
    print("   sebelum Cell 20, dan anotasi ulang validasi manusia (Cell 16-18).")


 ABLASI PROXY CLASSIFIER -- PROXY_ID = 0 
📌 Proxy aktif: [0] frozen_cls_lr — CLS embedding beku + Logistic Regression (P1)
 CONFIDENT LEARNING — proxy aktif: [0] frozen_cls_lr 
📥 Memuat 11375 baris data train.

🧮 Menghitung OOF pred_probs — proxy [0] frozen_cls_lr


config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/229k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

      Epoch 1/3 - Loss: 0.4904
      Epoch 2/3 - Loss: 0.4281
      Epoch 3/3 - Loss: 0.3671
   [Proxy finetuned_corn] Fold 5/5 (train=9100, val=2275)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

      Epoch 1/3 - Loss: 0.4869
      Epoch 2/3 - Loss: 0.4279
      Epoch 3/3 - Loss: 0.3659
   🌡️  Temperature scaling [finetuned_corn]: T=1.2074 (NLL 1.2837 -> 1.2439)

📐 Kualitas proxy [finetuned_corn]:
   Exact Accuracy : 0.4738
   MAE            : 0.7379
   Off-by-1 Acc   : 0.8422
   QWK            : 0.7083

🔎 Analisis Metode Filter Cleanlab:
   'confident_learning': 5332 baris diflag (46.87%)
   'prune_by_noise_rate': 4894 baris diflag (43.02%)

🎯 Pemilihan metode filter (kriteria: paling dekat dengan estimasi teoretis terkalibrasi):
   Estimasi teoretis (num_label_issues, off_diagonal_calibrated) : 6043 baris
   'confident_learning': 5332 baris (selisih: 711) <-- DIPILIH
   'prune_by_noise_rate': 4894 baris (selisih: 1149)

✅ Deteksi selesai (metode terpilih: confident_learning).
   Hard-prune     : buang 5332 / sisa 6043
   Severity-aware : buang 1595 / sisa 9780

📊 Distribusi rating_diff pada baris noise:
rating_diff
1    3737
2    1192
3     314
4      89
Name: count, dtype: 

,proxy_id,proxy_name,proxy_desc,accuracy,mae,off_by_one,qwk,pct_flagged_noise,selected_filter_method,n_issues_confident_learning,n_issues_prune_by_noise_rate,n_issues_estimated_theoretical
0,0,frozen_cls_lr,CLS embedding beku + Logistic Regression (P1),0.446681,0.888176,0.769055,0.635031,44.219780,confident_learning,5030,4394,6208
1,1,frozen_meanpool_lr,Mean-pooling embedding beku + Logistic Regress...,0.447736,0.881055,0.772396,0.638900,44.808791,confident_learning,5097,4429,6254
2,2,finetuned_ce,"IndoBERT fine-tuned K-Fold, CE loss (P3)",0.455912,0.738637,0.844484,0.700141,50.514286,confident_learning,5746,5036,6246
3,3,finetuned_corn,"IndoBERT fine-tuned K-Fold, CORN loss (P4) -- ...",0.473846,0.737934,0.842198,0.708326,46.874725,confident_learning,5332,4894,6043



🔍 Metode terpilih untuk proxy final (P4): confident_learning
   Kalau ini beda dari run sebelumnya, HAPUS experiment_progress.json
   sebelum Cell 20, dan anotasi ulang validasi manusia (Cell 16-18).


## Ablasi Tambahan — P5 (Fusi Sentimen Eksternal)

P5 **bukan** bagian dari pilot study Bab 1, tapi bagian dari studi ablasi lima
tahap penuh (Tabel 3.2) yang dilaporkan di Bab IV. Menambah satu baris ke
`proxy_ablation_table.csv` (proxy_id=4). Model fusion sekarang memproyeksikan
skor sentimen (3-dim) ke ruang 32-dim sebelum digabung dengan representasi
BERT (768-dim), supaya sinyalnya tidak tenggelam secara numerik (lihat
`models.py`).

⚠️ Cell ini **wajib diakhiri** dengan `config.set_proxy(3)` — supaya proxy aktif
kembali ke final (P4) sebelum cell manapun setelah ini dijalankan.

In [ ]:
# ==========================================================
# CELL 14: ABLASI TAMBAHAN -- P5 (FUSI SENTIMEN)
# ==========================================================
import traceback
from src import config
from src.clean import run_confident_learning

config.set_proxy(4)
try:
    df_noise_p5, proxy_metrics_p5 = run_confident_learning()
except Exception as e:
    print(f"⚠️ Proxy 4 (fusion) gagal:")
    traceback.print_exc()
finally:
    config.set_proxy(3)
    print(f"\n📌 Proxy dikembalikan ke final: [{config.PROXY_ID}] {config.PROXY_NAME}")

📌 Proxy aktif: [4] finetuned_corn_fusion — IndoBERT+CORN + fusi sentimen eksternal (P5)
 CONFIDENT LEARNING — proxy aktif: [4] finetuned_corn_fusion 
📥 Memuat 11375 baris data train.

🧮 Menghitung OOF pred_probs — proxy [4] finetuned_corn_fusion
⬇️  Memuat model sentimen: w11wo/indonesian-roberta-base-sentiment-classifier


config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/328 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/808k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/467k [00:00<?, ?B/s]

...

      Epoch 1/3 - Loss: 0.4929
      Epoch 2/3 - Loss: 0.4311
      Epoch 3/3 - Loss: 0.3714
   [Proxy finetuned_corn_fusion] Fold 4/5 (train=9100, val=2275)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

      Epoch 1/3 - Loss: 0.4929
      Epoch 2/3 - Loss: 0.4290
      Epoch 3/3 - Loss: 0.3682
   [Proxy finetuned_corn_fusion] Fold 5/5 (train=9100, val=2275)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

      Epoch 1/3 - Loss: 0.4877
      Epoch 2/3 - Loss: 0.4277
      Epoch 3/3 - Loss: 0.3722
   🌡️  Temperature scaling [finetuned_corn_fusion]: T=1.2154 (NLL 1.3003 -> 1.2576)

📐 Kualitas proxy [finetuned_corn_fusion]:
   Exact Accuracy : 0.4658
   MAE            : 0.7524
   Off-by-1 Acc   : 0.8382
   QWK            : 0.6968

🔎 Analisis Metode Filter Cleanlab:
   'confident_learning': 5484 baris diflag (48.21%)
   'prune_by_noise_rate': 4939 baris diflag (43.42%)

🎯 Pemilihan metode filter (kriteria: paling dekat dengan estimasi teoretis terkalibrasi):
   Estimasi teoretis (num_label_issues, off_diagonal_calibrated) : 6131 baris
   'confident_learning': 5484 baris (selisih: 647) <-- DIPILIH
   'prune_by_noise_rate': 4939 baris (selisih: 1192)

✅ Deteksi selesai (metode terpilih: confident_learning).
   Hard-prune     : buang 5484 / sisa 5891
   Severity-aware : buang 1651 / sisa 9724

📊 Distribusi rating_diff pada baris noise:
rating_diff
1    3833
2    1231
3     328
4      92
Name: 

## Kunci Proxy Final (P4) + Ekspor Sample Validasi Manusia

Menjalankan ulang Confident Learning dengan proxy final terkunci (P4/CORN,
sesuai skema integrasi ketat Bab 1.1) — akan memuat dari cache kalau sudah
pernah dihitung di Cell 12, jadi cepat. Ini menghasilkan sample 50 baris
(stratified by rating_diff) untuk validasi manusia.

⚠️ **BERHENTI setelah cell ini** — isi dulu `human_verdict` secara manual
sebelum lanjut ke Cell 17.

In [ ]:
# ==========================================================
# CELL 16: KUNCI PROXY FINAL (P4) -- WAJIB SEBELUM VALIDASI MANUSIA
# ==========================================================
from src import config

config.set_proxy(3)
print(f"🔒 Proxy final terkunci: [{config.PROXY_ID}] {config.PROXY_NAME}")

from src.clean import run_confident_learning
df_noise, proxy_metrics = run_confident_learning()

print("\n⚠️  BERHENTI DI SINI SEBELUM LANJUT KE CELL 17 ⚠️")
print(f"1. Buka: {config.HUMAN_VALIDATION_FILE}")
print("2. Isi kolom 'human_verdict' MANUAL untuk SEMUA baris:")
print("   'noise' / 'not_noise' / 'ambiguous'")
print("3. Save file (pastikan tersimpan di Drive, bukan di lokal komputer).")
print("4. Jalankan Cell 17 untuk cek kelengkapan + hitung agreement rate.")

📌 Proxy aktif: [3] finetuned_corn — IndoBERT fine-tuned K-Fold, CORN loss (P4) -- DEFAULT/FINAL
🔒 Proxy final terkunci: [3] finetuned_corn
 CONFIDENT LEARNING — proxy aktif: [3] finetuned_corn 
📥 Memuat 11375 baris data train.

🧮 Menghitung OOF pred_probs — proxy [3] finetuned_corn
⚡ Memuat cache proxy [finetuned_corn] ...

📐 Kualitas proxy [finetuned_corn]:
   Exact Accuracy : 0.4738
   MAE            : 0.7379
   Off-by-1 Acc   : 0.8422
   QWK            : 0.7083

🔎 Analisis Metode Filter Cleanlab:
   'confident_learning': 5332 baris diflag (46.87%)
   'prune_by_noise_rate': 4894 baris diflag (43.02%)

🎯 Pemilihan metode filter (kriteria: paling dekat dengan estimasi teoretis terkalibrasi):
   Estimasi teoretis (num_label_issues, off_diagonal_calibrated) : 6043 baris
   'confident_learning': 5332 baris (selisih: 711) <-- DIPILIH
   'prune_by_noise_rate': 4894 baris (selisih: 1149)

✅ Deteksi selesai (metode terpilih: confident_learning).
   Hard-prune     : buang 5332 / sisa 6043
   S

## Hitung Agreement Rate Validasi Manusia

Menjalankan **setelah** kamu selesai mengisi `human_verdict` secara manual di
file yang disebut Cell 16. Melaporkan agreement rate keseluruhan dan per-bin
`rating_diff` — breakdown per-bin ini penting sebagai bukti kualitatif
pendukung/penolak asumsi ambang severity-aware pruning (Subbab 3.7).

In [ ]:
from src import config
import pandas as pd

df_check = pd.read_csv(config.HUMAN_VALIDATION_FILE, sep=None, engine="python")
print(df_check.shape)
print(df_check["human_verdict"].value_counts(dropna=False))

(100, 7)
human_verdict
NaN    100
Name: count, dtype: int64


In [ ]:
# ==========================================================
# CELL 18: HITUNG AGREEMENT VALIDASI MANUSIA
# ==========================================================
from src.human_validation import compute_agreement

result = compute_agreement()
if result is not None:
    print("\n✅ Validasi manusia lengkap. Siap lanjut ke Cell 19 (training).")
else:
    print("\n⛔ Belum lengkap/ada error -- perbaiki dulu sebelum lanjut.")

   (file dibaca dengan delimiter ',')
 HASIL VALIDASI MANUSIA vs CLEANLAB 
Total sample direview : 100
Setuju (memang noise) : 16 (16.0%)
Ambigu                : 0 (0.0%)
Tidak setuju          : 84 (84.0%)

📊 Agreement rate per rating_diff:
   diff=1: n=70 | noise=11.4% | not_noise=88.6% | ambiguous=0.0%
   diff=2: n=22 | noise=18.2% | not_noise=81.8% | ambiguous=0.0%
   diff=3: n=6 | noise=33.3% | not_noise=66.7% | ambiguous=0.0%
   diff=4: n=2 | noise=100.0% | not_noise=0.0% | ambiguous=0.0%
------------------------------------------------------------
Acuan pembanding (Northcutt et al., 2021, ImageNet): ~58% sample
yang direview terbukti benar-benar issue -- acuan wajar, bukan standar mutlak.

💾 Hasil lengkap -> /content/drive/MyDrive/SKRIPSI_CORN/human_validation/human_validation_result__finetuned_corn.csv

✅ Validasi manusia lengkap. Siap lanjut ke Cell 19 (training).


In [ ]:
# ==========================================================
# CELL 18B: ANOTATOR KEDUA + COHEN'S KAPPA -- VALIDASI NOISE (Subbab 3.9.3)
# ==========================================================
from src import human_validation as hv

hv.export_second_annotator_subset()
print("\n⚠️  Minta penilai kedua mengisi 'human_verdict' pada file di atas, BUTA seperti anotator pertama.")
print("Setelah selesai, jalankan:")
print(">>> from src import human_validation as hv")
print(">>> hv.compute_interannotator_kappa()")
print("\nKalau hanya ada satu penilai (fallback single-annotator), jalankan sebagai gantinya:")
print(">>> hv.export_retest_subset()")
print(">>> # isi manual, lalu:")
print(">>> hv.compute_test_retest_reliability()")

📝 Sample anotator kedua (30 baris, 30% dari total) -> /content/drive/MyDrive/SKRIPSI_CORN/human_validation/human_validation_sample__finetuned_corn__annotator2.csv

⚠️  Minta penilai kedua mengisi 'human_verdict' pada file di atas, BUTA seperti anotator pertama.
Setelah selesai, jalankan:
>>> from src import human_validation as hv
>>> hv.compute_interannotator_kappa()

Kalau hanya ada satu penilai (fallback single-annotator), jalankan sebagai gantinya:
>>> hv.export_retest_subset()
>>> # isi manual, lalu:
>>> hv.compute_test_retest_reliability()


## Training 6 Skenario × 3 Seed

Melatih 6 skenario (kombinasi 3 varian data × 2 fungsi loss): M1-M3 (raw/
hard-prune/severity-prune × Cross-Entropy), M4-M6 (raw/hard-prune/severity-prune
× CORN). Setiap skenario dilatih pada 3 seed (42, 123, 2024) dengan
mixed-precision training dan early stopping (berhenti kalau Val MAE tidak
membaik 3 epoch berturut-turut).

Auto-resume: kalau sesi Colab terputus, jalankan ulang cell ini — training
akan melanjutkan dari skenario/seed terakhir yang belum selesai, bukan mulai
dari nol.

⏱️ Ini cell paling lama di seluruh notebook — bisa beberapa jam tergantung GPU
yang dialokasikan Colab.

In [13]:
# ==========================================================
# CELL 20: TRAINING 6 SKENARIO x 3 SEED (AUTO-RESUME)
# ==========================================================
import json
import numpy as np
from src.train import run_experiment
from src import config

print(f"🔧 Training M1-M6 akan pakai backbone: {config.PRETRAINED_MODEL_NAME} | "
      f"proxy final: {config.PROXY_NAME}")

scenarios = [
    {"name": "M1_Baseline_CE",        "train_path": config.TRAIN_RAW_FILE,           "loss": "ce"},
    {"name": "M2_CleanedHard_CE",     "train_path": config.TRAIN_CLEANED_HARD_FILE,  "loss": "ce"},
    {"name": "M3_CleanedSevere_CE",   "train_path": config.TRAIN_CLEANED_SEVERE_FILE,"loss": "ce"},
    {"name": "M4_Baseline_CORN",      "train_path": config.TRAIN_RAW_FILE,           "loss": "corn"},
    {"name": "M5_CleanedHard_CORN",   "train_path": config.TRAIN_CLEANED_HARD_FILE,  "loss": "corn"},
    {"name": "M6_CleanedSevere_CORN", "train_path": config.TRAIN_CLEANED_SEVERE_FILE,"loss": "corn"},
]

if os.path.exists(config.PROGRESS_FILE):
    with open(config.PROGRESS_FILE) as f:
        saved_progress = json.load(f)
    print("🔄 Progress sebelumnya ditemukan, melanjutkan yang belum selesai...")
else:
    saved_progress = {}
    print("🆕 Memulai training dari awal...")

all_results = []
print(f"\n🔥 {len(scenarios)} SKENARIO x {len(config.SEED_LIST)} SEED 🔥\n")

for scenario in scenarios:
    name = scenario["name"]
    metrics_list = {"mae": [], "rmse": [], "accuracy": [], "off_by_one": [], "qwk": []}
    saved_progress.setdefault(name, {})

    for seed in config.SEED_LIST:
        seed_key = str(seed)
        if seed_key in saved_progress[name]:
            print(f"⏩ {name} | seed {seed} (sudah selesai)")
            metrics = saved_progress[name][seed_key]
        else:
            metrics = run_experiment(name, scenario["train_path"], scenario["loss"], seed)
            saved_progress[name][seed_key] = metrics
            with open(config.PROGRESS_FILE, "w") as f:
                json.dump(saved_progress, f, indent=2)

        for k in metrics_list:
            metrics_list[k].append(metrics[k])

    all_results.append({
        "Model": name,
        "MAE (↓)": f"{np.mean(metrics_list['mae']):.4f} ± {np.std(metrics_list['mae']):.4f}",
        "RMSE (↓)": f"{np.mean(metrics_list['rmse']):.4f} ± {np.std(metrics_list['rmse']):.4f}",
        "Acc (↑)": f"{np.mean(metrics_list['accuracy']):.4f} ± {np.std(metrics_list['accuracy']):.4f}",
        "Off-by-1 (↑)": f"{np.mean(metrics_list['off_by_one']):.4f} ± {np.std(metrics_list['off_by_one']):.4f}",
        "QWK (↑)": f"{np.mean(metrics_list['qwk']):.4f} ± {np.std(metrics_list['qwk']):.4f}",
        "_raw_mae": np.mean(metrics_list["mae"]),
    })

df_final = pd.DataFrame(sorted(all_results, key=lambda x: x["_raw_mae"])).drop(columns=["_raw_mae"])
df_final.to_csv(config.FINAL_RESULTS_TABLE_FILE, index=False)

print("\n" + "=" * 100)
print(" 🏆 HASIL 6 SKENARIO (untuk Bab 4) 🏆")
print("=" * 100)
display(df_final)

🔧 Training M1-M6 akan pakai backbone: indobenchmark/indobert-base-p1 | proxy final: finetuned_corn
🔄 Progress sebelumnya ditemukan, melanjutkan yang belum selesai...

🔥 6 SKENARIO x 3 SEED 🔥

⏩ M1_Baseline_CE | seed 42 (sudah selesai)
⏩ M1_Baseline_CE | seed 123 (sudah selesai)

🚀 M1_Baseline_CE | Seed: 2024 | Loss: CE


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Epoch 1/8 - Loss: 1.2735 - Val MAE: 0.7724 | QWK: 0.6870 | Off-by-1: 0.8111 | Acc: 0.4886
Epoch 2/8 - Loss: 1.1182 - Val MAE: 0.7223 | QWK: 0.7042 | Off-by-1: 0.8445 | Acc: 0.4763
Epoch 3/8 - Loss: 0.9489 - Val MAE: 0.7496 | QWK: 0.6918 | Off-by-1: 0.8295 | Acc: 0.4851
Epoch 4/8 - Loss: 0.7218 - Val MAE: 0.7425 | QWK: 0.6921 | Off-by-1: 0.8330 | Acc: 0.4824
Epoch 5/8 - Loss: 0.4828 - Val MAE: 0.7522 | QWK: 0.6768 | Off-by-1: 0.8251 | Acc: 0.4868
   ⏹ Early stopping di epoch 5 (Val MAE tidak membaik 3x)
🏆 Test (dari model Val MAE terbaik=0.7223): MAE=0.7145 | RMSE=1.0879 | Acc=0.4694 | Off-by-1=0.8583 | QWK=0.7152

🚀 M2_CleanedHard_CE | Seed: 42 | Loss: CE


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Epoch 1/8 - Loss: 0.7908 - Val MAE: 0.3041 | QWK: 0.9057 | Off-by-1: 0.9537 | Acc: 0.7669
Epoch 2/8 - Loss: 0.4801 - Val MAE: 0.2645 | QWK: 0.9174 | Off-by-1: 0.9669 | Acc: 0.7901
Epoch 3/8 - Loss: 0.2910 - Val MAE: 0.2661 | QWK: 0.9187 | Off-by-1: 0.9603 | Acc: 0.7917
Epoch 4/8 - Loss: 0.1870 - Val MAE: 0.2860 | QWK: 0.9056 | Off-by-1: 0.9554 | Acc: 0.7868
Epoch 5/8 - Loss: 0.1110 - Val MAE: 0.2430 | QWK: 0.9340 | Off-by-1: 0.9736 | Acc: 0.7934
Epoch 6/8 - Loss: 0.0763 - Val MAE: 0.2959 | QWK: 0.8995 | Off-by-1: 0.9587 | Acc: 0.7769
Epoch 7/8 - Loss: 0.0626 - Val MAE: 0.3240 | QWK: 0.9032 | Off-by-1: 0.9570 | Acc: 0.7405
Epoch 8/8 - Loss: 0.0579 - Val MAE: 0.2942 | QWK: 0.9072 | Off-by-1: 0.9521 | Acc: 0.7736
   ⏹ Early stopping di epoch 8 (Val MAE tidak membaik 3x)
🏆 Test (dari model Val MAE terbaik=0.2430): MAE=0.7226 | RMSE=1.1209 | Acc=0.4786 | Off-by-1=0.8541 | QWK=0.7043

🚀 M2_CleanedHard_CE | Seed: 123 | Loss: CE


...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Epoch 1/8 - Loss: 0.4203 - Val MAE: 0.4969 | QWK: 0.8535 | Off-by-1: 0.9642 | Acc: 0.5470
Epoch 2/8 - Loss: 0.3444 - Val MAE: 0.4877 | QWK: 0.8622 | Off-by-1: 0.9519 | Acc: 0.5685
Epoch 3/8 - Loss: 0.2851 - Val MAE: 0.4826 | QWK: 0.8641 | Off-by-1: 0.9540 | Acc: 0.5695
Epoch 4/8 - Loss: 0.2208 - Val MAE: 0.4857 | QWK: 0.8657 | Off-by-1: 0.9611 | Acc: 0.5573
Epoch 5/8 - Loss: 0.1605 - Val MAE: 0.5010 | QWK: 0.8576 | Off-by-1: 0.9489 | Acc: 0.5613
Epoch 6/8 - Loss: 0.1168 - Val MAE: 0.5204 | QWK: 0.8496 | Off-by-1: 0.9540 | Acc: 0.5358
   ⏹ Early stopping di epoch 6 (Val MAE tidak membaik 3x)
🏆 Test (dari model Val MAE terbaik=0.4826): MAE=0.7391 | RMSE=1.1310 | Acc=0.4684 | Off-by-1=0.8453 | QWK=0.7040

🚀 M6_CleanedSevere_CORN | Seed: 2024 | Loss: CORN


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Epoch 1/8 - Loss: 0.4210 - Val MAE: 0.5112 | QWK: 0.8532 | Off-by-1: 0.9407 | Acc: 0.5552
Epoch 2/8 - Loss: 0.3461 - Val MAE: 0.4980 | QWK: 0.8599 | Off-by-1: 0.9387 | Acc: 0.5675
Epoch 3/8 - Loss: 0.2799 - Val MAE: 0.4724 | QWK: 0.8722 | Off-by-1: 0.9601 | Acc: 0.5685
Epoch 4/8 - Loss: 0.2177 - Val MAE: 0.4867 | QWK: 0.8619 | Off-by-1: 0.9479 | Acc: 0.5716
Epoch 5/8 - Loss: 0.1571 - Val MAE: 0.5000 | QWK: 0.8538 | Off-by-1: 0.9499 | Acc: 0.5562
Epoch 6/8 - Loss: 0.1156 - Val MAE: 0.5460 | QWK: 0.8321 | Off-by-1: 0.9499 | Acc: 0.5082
   ⏹ Early stopping di epoch 6 (Val MAE tidak membaik 3x)
🏆 Test (dari model Val MAE terbaik=0.4724): MAE=0.6990 | RMSE=1.0749 | Acc=0.4817 | Off-by-1=0.8594 | QWK=0.7233

 🏆 HASIL 6 SKENARIO (untuk Bab 4) 🏆


,Model,MAE (↓),RMSE (↓),Acc (↑),Off-by-1 (↑),QWK (↑)
0,M6_CleanedSevere_CORN,0.7114 ± 0.0196,1.0874 ± 0.0317,0.4742 ± 0.0056,0.8565 ± 0.0083,0.7188 ± 0.0108
1,M2_CleanedHard_CE,0.7181 ± 0.0065,1.1069 ± 0.0110,0.4788 ± 0.0037,0.8509 ± 0.0027,0.7108 ± 0.0059
2,M4_Baseline_CORN,0.7223 ± 0.0023,1.0784 ± 0.0131,0.4546 ± 0.0073,0.8594 ± 0.0072,0.7103 ± 0.0040
3,M3_CleanedSevere_CE,0.7242 ± 0.0125,1.1175 ± 0.0161,0.4789 ± 0.0081,0.8463 ± 0.0080,0.7147 ± 0.0067
4,M5_CleanedHard_CORN,0.7261 ± 0.0144,1.1115 ± 0.0180,0.4726 ± 0.0063,0.8487 ± 0.0052,0.7132 ± 0.0074
5,M1_Baseline_CE,0.7339 ± 0.0170,1.1214 ± 0.0331,0.4718 ± 0.0031,0.8411 ± 0.0132,0.7061 ± 0.0066


## Uji Signifikansi — Wilcoxon + Holm-Bonferroni (4 Hipotesis)

Menguji 4 hipotesis pre-registered (Subbab 3.9.2):
- **H1**: CORN vs CE pada data mentah (M4 vs M1)
- **H2**: severity-aware vs baseline pada CORN (M6 vs M4)
- **H3**: hard-prune vs baseline pada CORN (M5 vs M4)
- **H4**: severity-aware vs hard-prune langsung (M6 vs M5) — menjawab langsung
  pertanyaan rumusan masalah Bab 1.3 soal mana strategi cleaning yang lebih baik,
  bukan disimpulkan tidak langsung dari H2 dan H3

Prediksi dikumpulkan dari **ketiga seed** (bukan cuma seed 42), diagregasi
per sampel, baru diuji — sesuai janji Subbab 3.9.2. Effect size + CI 95%
(bootstrap) dilaporkan sebagai pelengkap p-value, disimpan otomatis ke CSV.

In [14]:
# ==========================================================
# CELL 22: UJI SIGNIFIKANSI -- WILCOXON + HOLM-BONFERRONI (4 HIPOTESIS)
# ==========================================================
from src.significance import (
    collect_all_predictions,
    aggregate_errors_across_seeds,
    run_significance_test,
    run_all_effect_sizes,
)
from src import config

print("📥 Mengumpulkan prediksi dari 3 seed x 6 skenario...")
true_labels, preds_per_seed = collect_all_predictions(scenarios)

print("\n📊 Mengagregasi absolute error across seed...")
aggregated_errors = aggregate_errors_across_seeds(true_labels, preds_per_seed)

print("\n🔬 Uji Wilcoxon Signed-Rank (4 hipotesis pre-registered) + Holm-Bonferroni...")
sig_results = run_significance_test(aggregated_errors)
display(sig_results)

print("\n📐 Effect size + CI 95% (bootstrap) -- pelengkap p-value...")
effect_sizes = run_all_effect_sizes(aggregated_errors)
display(effect_sizes)

effect_sizes_file = os.path.join(config.RESULTS_DIR, "effect_sizes.csv")
effect_sizes.to_csv(effect_sizes_file, index=False)
print(f"\n💾 Effect sizes -> {effect_sizes_file}")

📥 Mengumpulkan prediksi dari 3 seed x 6 skenario...
   Memuat prediksi M1_Baseline_CE | seed 42 ...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

   Memuat prediksi M1_Baseline_CE | seed 123 ...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

   Memuat prediksi M1_Baseline_CE | seed 2024 ...


...


📊 Mengagregasi absolute error across seed...

🔬 Uji Wilcoxon Signed-Rank (4 hipotesis pre-registered) + Holm-Bonferroni...

💾 Hasil uji signifikansi (4 hipotesis, Holm-Bonferroni) -> /content/drive/MyDrive/SKRIPSI_CORN/results/significance_test.csv


,hypothesis,model_a,model_b,mean_error_a,mean_error_b,p_value_raw,p_value_corrected,signifikan
0,H1_CORN_vs_CE_raw,M4_Baseline_CORN,M1_Baseline_CE,0.722339,0.733943,0.194308,0.582923,Tidak
1,H2_SeverityAware_vs_Base,M6_CleanedSevere_CORN,M4_Baseline_CORN,0.711439,0.722339,0.205330,0.582923,Tidak
2,H3_HardPrune_vs_Base,M5_CleanedHard_CORN,M4_Baseline_CORN,0.726090,0.722339,0.854051,0.854051,Tidak
3,H4_SeverityAware_vs_Hard,M6_CleanedSevere_CORN,M5_CleanedHard_CORN,0.711439,0.726090,0.060817,0.243269,Tidak



📐 Effect size + CI 95% (bootstrap) -- pelengkap p-value...


,model_a,model_b,mean_diff,ci_95_low,ci_95_high
0,M4_Baseline_CORN,M1_Baseline_CE,-0.011603,-0.024968,0.001406
1,M6_CleanedSevere_CORN,M4_Baseline_CORN,-0.010900,-0.024265,0.002696
2,M5_CleanedHard_CORN,M4_Baseline_CORN,0.003751,-0.012072,0.018519
3,M6_CleanedSevere_CORN,M5_CleanedHard_CORN,-0.014651,-0.028481,-0.001755



💾 Effect sizes -> /content/drive/MyDrive/SKRIPSI_CORN/results/effect_sizes.csv


In [16]:
# ==========================================================
# CELL 23: EVALUASI 6 SKENARIO PADA SUBSET UJI EMAS -- Subbab 3.9.4
# ==========================================================
from src import config, gold_test

result = gold_test.evaluate_on_gold_subset(scenarios)
if result is not None:
    print("\n✅ Evaluasi subset uji emas selesai -- lampirkan ke Bab IV berdampingan "
          "dengan hasil data uji penuh (Cell 20 & 22).")
    gold_test.compute_gold_interannotator_kappa()
else:
    print(f"\n⛔ Lengkapi dulu {config.GOLD_TEST_SAMPLE_FILE} sebelum menjalankan cell ini.")

🏅 Evaluasi 6 skenario pada subset uji emas (n=93).


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


💾 Hasil evaluasi subset uji emas -> /content/drive/MyDrive/SKRIPSI_CORN/results/gold_test_evaluation.csv
                Model      MAE     RMSE  Accuracy  Off_by_one      QWK  n_gold
     M4_Baseline_CORN 0.544803 0.813609  0.508961    0.953405 0.791120      93
    M2_CleanedHard_CE 0.555556 0.839737  0.516129    0.931900 0.787920      93
M6_CleanedSevere_CORN 0.580645 0.872208  0.498208    0.931900 0.771674      93
  M5_CleanedHard_CORN 0.584229 0.896503  0.512545    0.913978 0.771075      93
  M3_CleanedSevere_CE 0.602151 0.881755  0.476703    0.928315 0.777537      93
       M1_Baseline_CE 0.605735 0.912171  0.498208    0.906810 0.757049      93

💾 Arah & effect size subset uji emas -> /content/drive/MyDrive/SKRIPSI_CORN/results/gold_test_effect_sizes.csv
              hypothesis               model_a             model_b  mean_diff  ci_95_low  ci_95_high         arah
       H1_CORN_vs_CE_raw      M4_Baseline_CORN      M1_Baseline_CE  -0.060932  -0.146953    0.025090 A lebih baik
H

## Ringkasan Akhir — Siap Disalin ke Bab 1 (Pilot Study) dan Bab 4

Semua dibaca dari file di Drive (bukan variabel session), jadi cell ini bisa
dijalankan kapan saja, tidak peduli cell mana yang sudah/belum jalan di sesi
Colab saat ini.

In [17]:
# ==========================================================
# CELL 24: RINGKASAN AKHIR
# ==========================================================
import pandas as pd
from src import config

print("=" * 70)
print(" RINGKASAN LENGKAP ")
print("=" * 70)

print("\n[1] TABEL ABLASI PROXY (P1-P5) -- untuk Bab 1 (pilot study P1-P4) & Bab 4 (lengkap P1-P5):")
if os.path.exists(config.PROXY_QUALITY_LOG_FILE):
    display(pd.read_csv(config.PROXY_QUALITY_LOG_FILE))
else:
    print("   ⚠️ Belum ada -- jalankan Cell 12 dulu.")

print("\n[2] LAPORAN KUALITAS DATA SCRAPING -- untuk Bab 3:")
for fname in ["scraping_summary.csv", "date_distribution_per_app_rating.csv",
              "text_length_summary.csv", "duplicate_review_texts.csv"]:
    fpath = os.path.join(config.RESULTS_DIR, fname)
    status = "✅" if os.path.exists(fpath) else "⚠️ belum ada"
    print(f"   {status} {fpath}")

print("\n[3] VALIDASI MANUSIA -- untuk Bab 4:")
if os.path.exists(config.HUMAN_VALIDATION_RESULT_FILE):
    df_human = pd.read_csv(config.HUMAN_VALIDATION_RESULT_FILE)
    counts = df_human["human_verdict"].value_counts()
    total = len(df_human)
    print(f"   Total: {total} | Noise: {counts.get('noise',0)} ({counts.get('noise',0)/total*100:.1f}%) "
          f"| Not noise: {counts.get('not_noise',0)} ({counts.get('not_noise',0)/total*100:.1f}%) "
          f"| Ambiguous: {counts.get('ambiguous',0)} ({counts.get('ambiguous',0)/total*100:.1f}%)")
else:
    print("   ⚠️ Belum ada -- jalankan Cell 18 dulu.")

print("\n[4] HASIL 6 SKENARIO (Mean ± Std, 3 seed) -- untuk Bab 4:")
if os.path.exists(config.FINAL_RESULTS_TABLE_FILE):
    display(pd.read_csv(config.FINAL_RESULTS_TABLE_FILE))
else:
    print("   ⚠️ Belum ada -- jalankan Cell 20 dulu.")

print("\n[5] UJI SIGNIFIKANSI (4 hipotesis pre-registered) -- untuk Bab 4:")
if os.path.exists(config.SIGNIFICANCE_TEST_FILE):
    display(pd.read_csv(config.SIGNIFICANCE_TEST_FILE))
else:
    print("   ⚠️ Belum ada -- jalankan Cell 22 dulu.")

print("\n[6] EFFECT SIZE + CI 95% -- untuk Bab 4:")
effect_size_file = os.path.join(config.RESULTS_DIR, "effect_sizes.csv")
if os.path.exists(effect_size_file):
    display(pd.read_csv(effect_size_file))
else:
    print("   ⚠️ Belum ada -- jalankan Cell 22 dulu.")

print("\n✅ Semua file hasil tersimpan di:", config.RESULTS_DIR)

 RINGKASAN LENGKAP 

[1] TABEL ABLASI PROXY (P1-P5) -- untuk Bab 1 (pilot study P1-P4) & Bab 4 (lengkap P1-P5):


,proxy_id,proxy_name,proxy_desc,accuracy,mae,off_by_one,qwk,pct_flagged_noise,selected_filter_method,n_issues_confident_learning,n_issues_prune_by_noise_rate,n_issues_estimated_theoretical
0,0,frozen_cls_lr,CLS embedding beku + Logistic Regression (P1),0.446681,0.888176,0.769055,0.635031,44.219780,confident_learning,5030,4394,6208
1,1,frozen_meanpool_lr,Mean-pooling embedding beku + Logistic Regress...,0.447736,0.881055,0.772396,0.638900,44.808791,confident_learning,5097,4429,6254
2,2,finetuned_ce,"IndoBERT fine-tuned K-Fold, CE loss (P3)",0.455912,0.738637,0.844484,0.700141,50.514286,confident_learning,5746,5036,6246
3,3,finetuned_corn,"IndoBERT fine-tuned K-Fold, CORN loss (P4) -- ...",0.473846,0.737934,0.842198,0.708326,46.874725,confident_learning,5332,4894,6043
4,4,finetuned_corn_fusion,IndoBERT+CORN + fusi sentimen eksternal (P5),0.465758,0.752352,0.838242,0.696761,48.210989,confident_learning,5484,4939,6131



[2] LAPORAN KUALITAS DATA SCRAPING -- untuk Bab 3:
   ⚠️ belum ada /content/drive/MyDrive/SKRIPSI_CORN/results/scraping_summary.csv
   ⚠️ belum ada /content/drive/MyDrive/SKRIPSI_CORN/results/date_distribution_per_app_rating.csv
   ⚠️ belum ada /content/drive/MyDrive/SKRIPSI_CORN/results/text_length_summary.csv
   ⚠️ belum ada /content/drive/MyDrive/SKRIPSI_CORN/results/duplicate_review_texts.csv

[3] VALIDASI MANUSIA -- untuk Bab 4:
   Total: 100 | Noise: 16 (16.0%) | Not noise: 84 (84.0%) | Ambiguous: 0 (0.0%)

[4] HASIL 6 SKENARIO (Mean ± Std, 3 seed) -- untuk Bab 4:


,Model,MAE (↓),RMSE (↓),Acc (↑),Off-by-1 (↑),QWK (↑)
0,M6_CleanedSevere_CORN,0.7114 ± 0.0196,1.0874 ± 0.0317,0.4742 ± 0.0056,0.8565 ± 0.0083,0.7188 ± 0.0108
1,M2_CleanedHard_CE,0.7181 ± 0.0065,1.1069 ± 0.0110,0.4788 ± 0.0037,0.8509 ± 0.0027,0.7108 ± 0.0059
2,M4_Baseline_CORN,0.7223 ± 0.0023,1.0784 ± 0.0131,0.4546 ± 0.0073,0.8594 ± 0.0072,0.7103 ± 0.0040
3,M3_CleanedSevere_CE,0.7242 ± 0.0125,1.1175 ± 0.0161,0.4789 ± 0.0081,0.8463 ± 0.0080,0.7147 ± 0.0067
4,M5_CleanedHard_CORN,0.7261 ± 0.0144,1.1115 ± 0.0180,0.4726 ± 0.0063,0.8487 ± 0.0052,0.7132 ± 0.0074
5,M1_Baseline_CE,0.7339 ± 0.0170,1.1214 ± 0.0331,0.4718 ± 0.0031,0.8411 ± 0.0132,0.7061 ± 0.0066



[5] UJI SIGNIFIKANSI (4 hipotesis pre-registered) -- untuk Bab 4:


,hypothesis,model_a,model_b,mean_error_a,mean_error_b,p_value_raw,p_value_corrected,signifikan
0,H1_CORN_vs_CE_raw,M4_Baseline_CORN,M1_Baseline_CE,0.722339,0.733943,0.194308,0.582923,Tidak
1,H2_SeverityAware_vs_Base,M6_CleanedSevere_CORN,M4_Baseline_CORN,0.711439,0.722339,0.205330,0.582923,Tidak
2,H3_HardPrune_vs_Base,M5_CleanedHard_CORN,M4_Baseline_CORN,0.726090,0.722339,0.854051,0.854051,Tidak
3,H4_SeverityAware_vs_Hard,M6_CleanedSevere_CORN,M5_CleanedHard_CORN,0.711439,0.726090,0.060817,0.243269,Tidak



[6] EFFECT SIZE + CI 95% -- untuk Bab 4:


,model_a,model_b,mean_diff,ci_95_low,ci_95_high
0,M4_Baseline_CORN,M1_Baseline_CE,-0.011603,-0.024968,0.001406
1,M6_CleanedSevere_CORN,M4_Baseline_CORN,-0.010900,-0.024265,0.002696
2,M5_CleanedHard_CORN,M4_Baseline_CORN,0.003751,-0.012072,0.018519
3,M6_CleanedSevere_CORN,M5_CleanedHard_CORN,-0.014651,-0.028481,-0.001755



✅ Semua file hasil tersimpan di: /content/drive/MyDrive/SKRIPSI_CORN/results
